## Download the Dataset

In [1]:
!pip install kaggle

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
 82% 21.0M/25.7M [00:00<00:00, 97.9MB/s]
100% 25.7M/25.7M [00:00<00:00, 97.5MB/s]


In [4]:
from zipfile import ZipFile
dataset = '/content/imdb-dataset-of-50k-movie-reviews.zip'
with ZipFile(dataset,'r') as zip:
  zip.extractall()
  print('Done')

Done


In [5]:
!ls

'IMDB Dataset.csv'   imdb-dataset-of-50k-movie-reviews.zip   kaggle.json   sample_data


## Data collection

In [6]:
import os
import json

import pandas as pd


In [7]:
data = pd.read_csv('/content/IMDB Dataset.csv')

In [8]:
data.shape

(50000, 2)

In [9]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [10]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [16]:
data.replace({"sentiment" : {"positive": 1,"negative": 0}},inplace=True)

# if you want to store this value into another variable i.e. data you need to
# add inplace=True at the end

<ipython-input-16-40d715b0aa58>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment" : {"positive": 1,"negative": 0}},inplace=True)


In [17]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train_data, test_data = train_test_split(data,test_size=0.2,random_state=42)

In [20]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


## Data preprocessing

In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [23]:
#Tokenize text data


tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

#sequences_train = tokenizer.texts_to_sequences(train_data["review"])
#sequences_test = tokenizer.texts_to_sequences(test_data["review"])
#X_train = pad_sequences(sequences_train,maxlen=200)
#X_test = pad_sequences(sequences_test,maxlen=200)


X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)




In [24]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [25]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [26]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [27]:
print(Y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


## Building an LSTM model

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

In [31]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

# Here in embedding as you can see inputdim is 5000 it comes from num_words
# which is use in tokenizer (this means the model expects a vocabulary of 5000 unique words)

# we had also written input_length=200 which comes from length which we write
# in pad_sequences Each word will be represented by a 128-dimensional vector.

# droupout is use for preventing overfitting and for genarlizing the model

In [32]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 200, 128)          640000    
                                                                 
 lstm_1 (LSTM)               (None, 128)               131584    
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 771713 (2.94 MB)
Trainable params: 771713 (2.94 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [33]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [34]:
model.fit(X_train, Y_train, batch_size=64, epochs=5, validation_data=(X_test, Y_test),validation_split=0.2)

Epoch 1/5
625/625 [==============================] - 294s 467ms/step - loss: 0.3979 - accuracy: 0.8216 - val_loss: 0.3006 - val_accuracy: 0.8769
Epoch 2/5
625/625 [==============================] - 284s 454ms/step - loss: 0.2774 - accuracy: 0.8871 - val_loss: 0.2961 - val_accuracy: 0.8817
Epoch 3/5
625/625 [==============================] - 283s 452ms/step - loss: 0.2281 - accuracy: 0.9107 - val_loss: 0.2938 - val_accuracy: 0.8883
Epoch 4/5
625/625 [==============================] - 283s 453ms/step - loss: 0.1885 - accuracy: 0.9269 - val_loss: 0.2945 - val_accuracy: 0.8893
Epoch 5/5
625/625 [==============================] - 283s 453ms/step - loss: 0.1641 - accuracy: 0.9371 - val_loss: 0.3609 - val_accuracy: 0.8829


## Model Evaluation

In [35]:
loss, accuracy = model.evaluate(X_test, Y_test)
print("Loss:", loss)
print("Accuracy:", accuracy)



313/313 [==============================] - 16s 51ms/step - loss: 0.3609 - accuracy: 0.8829
Loss: 0.36091309785842896
Accuracy: 0.8828999996185303


## Predictive System

In [46]:
def predict_sentiment():
  review = input("Enter your movie review: ")  # Get user input
  sequences = tokenizer.texts_to_sequences([review])
  padded_sequences = pad_sequences(sequences, maxlen=200)
  prediction = model.predict(padded_sequences)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  print(f"The sentiment of your review is: {sentiment}")

predict_sentiment()

# here we use prediction[0][0] becausethe output of model.predict is a two dimensional array
# prediction would be: [[0.8]]
# prediction[0] would give you: [0.8] (an array)
# prediction[0][0] would give you: 0.8 (the actual probability value)

Enter your movie review: Food quality is not good but location is good and rooms are big enough compared to other hostel.
1/1 [==============================] - 0s 43ms/step
The sentiment of your review is: positive
